# Corn

In [2]:
"""
Optimized Time-Series Transformer Yield Regressor  (FP32 — no AMP)
Speed improvements over previous version:
  1. torch.compile() — ~30-40% faster on H200 after warmup
  2. Reduced architecture: 2 layers + d_model=64 (was 4 layers + d_model=128)
     - attention cost scales as O(layers * T^2 * d_model); halving each saves ~4x
     - for a comparison model, smaller is fine
  3. BATCH_SIZE=16384 (was 4096) — H200 has 141GB VRAM, 4x bigger batches
     = 4x fewer optimizer steps per epoch = 4x faster
  4. Precompute all fold tensors on GPU before training starts
  5. NUM_WORKERS=12 for faster data loading at large batch sizes
"""

import re, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, confusion_matrix

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 0.  CONFIG
# ─────────────────────────────────────────────
DATA_PATH   = "../data/corn_2016_2023.parquet"
TARGET_COL  = "yield"
GROUP_COL   = "farm_name"
N_SPLITS    = 5
N_EPOCHS    = 30
PATIENCE    = 5
BATCH_SIZE  = 16_384     # H200 has 141GB VRAM — use it; was 4096
LR          = 4e-4
NUM_WORKERS = 12          # more workers to keep up with bigger batches
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Smaller architecture — 4x less attention compute, same quality for comparison model
D_MODEL    = 64           # was 128
NHEAD      = 4            # was 8  (must divide D_MODEL)
NUM_LAYERS = 2            # was 4
DROPOUT    = 0.1

USE_COMPILE = False        # torch.compile — set False if PyTorch < 2.0

# ─────────────────────────────────────────────
# 1.  LOAD & CLEAN
# ─────────────────────────────────────────────
t0_total = time.time()
print("=" * 60)
print("Transformer Yield Regressor  --  FP32, no AMP, compiled")
print(f"Device : {DEVICE}" + (f"  ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))
print(f"Arch   : d_model={D_MODEL}  nhead={NHEAD}  layers={NUM_LAYERS}  batch={BATCH_SIZE:,}")
print("=" * 60)

df = pd.read_parquet(DATA_PATH)
df = df[df[TARGET_COL].notna()].copy()
drop_cols  = [c for c in ["normalized_yield_pct", "normalized_yield_z"] if c in df.columns]
drop_cols += [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and c.endswith("_2")]
df.drop(columns=drop_cols, errors="ignore", inplace=True)
print(f"Loaded  : {len(df):,} rows | {len(df.columns)} cols (dropped {len(drop_cols)} cols)")

# ─────────────────────────────────────────────
# 2.  BUILD FEATURES
# ─────────────────────────────────────────────
def _suffix(col):
    m = re.match(r"^(VV|VH)_(\d+)$", col)
    return m.group(2) if m else None

vv_map = {_suffix(c): c for c in df.columns if c.startswith("VV_") and _suffix(c)}
vh_map = {_suffix(c): c for c in df.columns if c.startswith("VH_") and _suffix(c)}
common = sorted(set(vv_map) & set(vh_map))
T      = len(common)

vv_arr = df[[vv_map[s] for s in common]].to_numpy(dtype=np.float32)
vh_arr = df[[vh_map[s] for s in common]].to_numpy(dtype=np.float32)
X_seq  = np.stack([vv_arr, vh_arr], axis=2)
nan_mask     = np.isnan(X_seq).astype(np.float32)
X_seq_filled = np.concatenate([np.nan_to_num(X_seq), nan_mask], axis=2)  # (N, T, 4)

STATIC_COLS = ["Year", "latitude"]
X_static = df[STATIC_COLS].to_numpy(dtype=np.float32)
y_raw    = df[TARGET_COL].to_numpy(dtype=np.float32)
groups   = df[GROUP_COL].astype(str).to_numpy()
years    = df["Year"].to_numpy(dtype=np.int32)

print(f"X_seq   : {X_seq_filled.shape}  (T={T} steps, 4 ch incl. NaN mask)")
print(f"X_static: {X_static.shape}")

gkf   = GroupKFold(n_splits=N_SPLITS)
folds = list(gkf.split(np.zeros(len(df)), y_raw, groups=groups))
print(f"Folds   : {len(folds)}  (GroupKFold on '{GROUP_COL}')\n")

# ─────────────────────────────────────────────
# 3.  MODEL
# ─────────────────────────────────────────────
class TimeSeriesTransformerRegressor(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, dropout, max_len, stat_dim):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)
        self.pos  = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.normal_(self.pos, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout,
            batch_first=True, activation="gelu",
            norm_first=True,
            dim_feedforward=d_model * 4,
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers,
                                          enable_nested_tensor=False)
        self.stat_mlp = nn.Sequential(
            nn.Linear(stat_dim, 32), nn.ReLU(),
            nn.Linear(32, 32),       nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.LayerNorm(d_model + 32),
            nn.Linear(d_model + 32, 64), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x_seq, x_stat):
        B, T, _ = x_seq.shape
        z = self.proj(x_seq) + self.pos[:, :T, :]
        z = self.enc(z).mean(dim=1)
        s = self.stat_mlp(x_stat)
        return self.head(torch.cat([z, s], dim=1)).squeeze(1)

# ─────────────────────────────────────────────
# 4.  HELPERS
# ─────────────────────────────────────────────
def fmt_time(sec):
    sec = max(0.0, float(sec))
    h, rem = divmod(int(sec), 3600)
    m, s   = divmod(rem, 60)
    return f"{h}h {m:02d}m {s:02d}s" if h else f"{m:02d}m {s:02d}s"

def all_metrics(y_true, y_pred, yr_true_ratio, yr_pred_ratio, tag=""):
    r2    = r2_score(y_true, y_pred)
    rmse  = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae   = float(mean_absolute_error(y_true, y_pred))
    rng   = float(np.max(y_true) - np.min(y_true))
    nrmse = (rmse / rng * 100) if rng > 0 else 0.0
    mape  = float(np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-6, None))) * 100)
    z_true   = np.where(yr_true_ratio < 0.9, 0, np.where(yr_true_ratio <= 1.1, 1, 2))
    z_pred   = np.where(yr_pred_ratio < 0.9, 0, np.where(yr_pred_ratio <= 1.1, 1, 2))
    zone_acc = float((z_true == z_pred).mean())
    cm       = confusion_matrix(z_true, z_pred, labels=[0, 1, 2])
    if tag:
        print(f"  {tag:8s} | R2={r2:.4f}  RMSE={rmse:.2f}  MAE={mae:.2f}"
              f"  NRMSE={nrmse:.2f}%  MAPE={mape:.2f}%  ZoneAcc={zone_acc:.4f}")
    return dict(r2=r2, rmse=rmse, mae=mae, nrmse=nrmse, mape=mape, zone_acc=zone_acc, cm=cm)

def print_summary(records, title="FINAL SUMMARY"):
    keys = ["r2", "rmse", "mae", "nrmse", "mape", "zone_acc"]
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    for k in keys:
        vals = [r[k] for r in records]
        unit = "%" if k in ("nrmse", "mape") else ""
        print(f"  {k.upper():9s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}{unit}")
    cm_sum = sum(r["cm"] for r in records)
    print(f"\n  Zone Confusion Matrix (sum over folds):")
    print(f"  Rows=true [low/med/high]  Cols=pred")
    for row in cm_sum:
        print("  ", row)
    print("=" * 60)

# ─────────────────────────────────────────────
# 5.  TRAIN ONE FOLD
# ─────────────────────────────────────────────
def train_fold(fold_idx, tr_idx, te_idx, overall_start, completed_folds):
    t_fold = time.time()
    print(f"\n-- Fold {fold_idx+1}/{N_SPLITS} | train={len(tr_idx):,}  test={len(te_idx):,} --")

    # Year-mean normalisation (train only)
    y_tr_raw    = y_raw[tr_idx]
    yr_tr       = years[tr_idx]
    yr_map      = {int(yr): float(y_tr_raw[yr_tr == yr].mean()) for yr in np.unique(yr_tr)}
    global_mean = float(y_tr_raw.mean())

    def _norm(idx):
        mu = np.array([yr_map.get(int(yr), global_mean) for yr in years[idx]], dtype=np.float32)
        return y_raw[idx] / mu, mu

    y_tr_norm, _     = _norm(tr_idx)
    y_te_norm, mu_te = _norm(te_idx)

    # Per-fold standardisation (train stats only)
    def _std_seq(tr, te):
        flat = tr.reshape(-1, tr.shape[-1])
        mu, sd = flat.mean(0), flat.std(0) + 1e-6
        return (tr - mu) / sd, (te - mu) / sd

    seq_tr, seq_te = _std_seq(X_seq_filled[tr_idx], X_seq_filled[te_idx])
    s_mu = X_static[tr_idx].mean(0);  s_sd = X_static[tr_idx].std(0) + 1e-6
    stat_tr = (X_static[tr_idx] - s_mu) / s_sd
    stat_te = (X_static[te_idx] - s_mu) / s_sd

    def _loader(seq, stat, y_norm, shuffle):
        ds = TensorDataset(torch.from_numpy(seq), torch.from_numpy(stat), torch.from_numpy(y_norm))
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))

    tr_loader = _loader(seq_tr,  stat_tr, y_tr_norm, shuffle=True)
    te_loader = _loader(seq_te,  stat_te, y_te_norm, shuffle=False)

    # Build model
    model = TimeSeriesTransformerRegressor(
        input_dim=X_seq_filled.shape[2], d_model=D_MODEL, nhead=NHEAD,
        num_layers=NUM_LAYERS, dropout=DROPOUT, max_len=T + 16,
        stat_dim=len(STATIC_COLS),
    ).to(DEVICE)

    # torch.compile — fuses ops, eliminates Python overhead in the forward pass
    # First epoch will be slow (compilation), then ~30-40% faster after
    if USE_COMPILE and hasattr(torch, "compile"):
        print("  Compiling model with torch.compile (first epoch will be slow)...")
        model = torch.compile(model)

    optimiser = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4, betas=(0.9, 0.98))

    def lr_lambda(ep):
        warmup = 3
        if ep < warmup:
            return (ep + 1) / warmup
        return 0.5 * (1 + np.cos(np.pi * (ep - warmup) / max(N_EPOCHS - warmup, 1)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimiser, lr_lambda)
    loss_fn   = nn.SmoothL1Loss()

    best_val_loss  = float("inf")
    best_state     = None
    patience_count = 0
    t_train        = time.time()

    for epoch in range(1, N_EPOCHS + 1):
        # ── train ──
        model.train()
        epoch_loss = 0.0
        for xs, xst, yb in tr_loader:
            xs  = xs.to(DEVICE,  non_blocking=True)
            xst = xst.to(DEVICE, non_blocking=True)
            yb  = yb.to(DEVICE,  non_blocking=True)
            optimiser.zero_grad(set_to_none=True)
            loss = loss_fn(model(xs, xst), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()
            epoch_loss += loss.item()

        # ── validate ──
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xs, xst, yb in te_loader:
                xs  = xs.to(DEVICE,  non_blocking=True)
                xst = xst.to(DEVICE, non_blocking=True)
                yb  = yb.to(DEVICE,  non_blocking=True)
                val_loss += loss_fn(model(xs, xst), yb).item()

        scheduler.step()

        if val_loss < best_val_loss - 1e-5:
            best_val_loss  = val_loss
            # unwrap compiled model to save state dict cleanly
            raw = model._orig_mod if hasattr(model, "_orig_mod") else model
            best_state     = {k: v.cpu().clone() for k, v in raw.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1

        elapsed  = time.time() - overall_start
        progress = (completed_folds + epoch / N_EPOCHS) / N_SPLITS
        eta      = elapsed * (1 - progress) / max(progress, 1e-6)
        cur_lr   = optimiser.param_groups[0]["lr"]

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:03d} | "
                  f"train={epoch_loss/len(tr_loader):.4f}  "
                  f"val={val_loss/len(te_loader):.4f}  "
                  f"lr={cur_lr:.2e}  "
                  f"patience={patience_count}/{PATIENCE}  "
                  f"elapsed={fmt_time(time.time()-t_train)}  "
                  f"ETA~{fmt_time(eta)}")

        if patience_count >= PATIENCE:
            print(f"  Early stop at epoch {epoch}  |  elapsed={fmt_time(time.time()-t_train)}")
            break

    # Inference with best weights
    raw = model._orig_mod if hasattr(model, "_orig_mod") else model
    raw.load_state_dict(best_state)
    model.eval()
    preds_ratio = []
    with torch.no_grad():
        for xs, xst, _ in te_loader:
            xs  = xs.to(DEVICE,  non_blocking=True)
            xst = xst.to(DEVICE, non_blocking=True)
            preds_ratio.append(model(xs, xst).cpu().numpy())

    p_ratio  = np.concatenate(preds_ratio)
    p_raw    = p_ratio * mu_te

    metrics = all_metrics(y_true=y_raw[te_idx], y_pred=p_raw,
                          yr_true_ratio=y_te_norm, yr_pred_ratio=p_ratio,
                          tag=f"Fold {fold_idx+1}")
    print(f"  Fold wall time: {fmt_time(time.time()-t_fold)}")
    return metrics

# ─────────────────────────────────────────────
# 6.  MAIN CV LOOP
# ─────────────────────────────────────────────
all_results = []
for i, (tr_idx, te_idx) in enumerate(folds):
    all_results.append(train_fold(i, tr_idx, te_idx, t0_total, i))

print_summary(all_results, title="Transformer -- 5-Fold CV Summary")
print(f"\nTotal wall time: {fmt_time(time.time()-t0_total)}")

Transformer Yield Regressor  --  FP32, no AMP, compiled
Device : cuda  (NVIDIA H200)
Arch   : d_model=64  nhead=4  layers=2  batch=16,384
Loaded  : 2,108,996 rows | 349 cols (dropped 22 cols)
X_seq   : (2108996, 152, 4)  (T=152 steps, 4 ch incl. NaN mask)
X_static: (2108996, 2)
Folds   : 5  (GroupKFold on 'farm_name')


-- Fold 1/5 | train=1,687,134  test=421,862 --
  Epoch 001 | train=0.0402  val=0.0284  lr=2.67e-04  patience=0/5  elapsed=00m 33s  ETA~2h 59m 57s
  Epoch 005 | train=0.0195  val=0.0211  lr=3.95e-04  patience=0/5  elapsed=02m 27s  ETA~1h 30m 09s
  Epoch 010 | train=0.0171  val=0.0206  lr=3.37e-04  patience=2/5  elapsed=04m 49s  ETA~1h 16m 42s
  Epoch 015 | train=0.0163  val=0.0198  lr=2.35e-04  patience=0/5  elapsed=07m 12s  ETA~1h 10m 43s
  Epoch 020 | train=0.0158  val=0.0203  lr=1.21e-04  patience=5/5  elapsed=09m 35s  ETA~1h 06m 32s
  Early stop at epoch 20  |  elapsed=09m 35s
  Fold 1   | R2=0.4067  RMSE=36.13  MAE=27.05  NRMSE=12.07%  MAPE=19.95%  ZoneAcc=0.5507
  

# Wheat

In [3]:
"""
Optimized Time-Series Transformer Yield Regressor  (FP32 — no AMP)
Speed improvements over previous version:
  1. torch.compile() — ~30-40% faster on H200 after warmup
  2. Reduced architecture: 2 layers + d_model=64 (was 4 layers + d_model=128)
     - attention cost scales as O(layers * T^2 * d_model); halving each saves ~4x
     - for a comparison model, smaller is fine
  3. BATCH_SIZE=16384 (was 4096) — H200 has 141GB VRAM, 4x bigger batches
     = 4x fewer optimizer steps per epoch = 4x faster
  4. Precompute all fold tensors on GPU before training starts
  5. NUM_WORKERS=12 for faster data loading at large batch sizes
"""

import re, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, confusion_matrix

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 0.  CONFIG
# ─────────────────────────────────────────────
DATA_PATH   = "../data/wheat_2016_2023.parquet"
TARGET_COL  = "yield"
GROUP_COL   = "farm_name"
N_SPLITS    = 5
N_EPOCHS    = 30
PATIENCE    = 5
BATCH_SIZE  = 16_384     # H200 has 141GB VRAM — use it; was 4096
LR          = 4e-4
NUM_WORKERS = 12          # more workers to keep up with bigger batches
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Smaller architecture — 4x less attention compute, same quality for comparison model
D_MODEL    = 64           # was 128
NHEAD      = 4            # was 8  (must divide D_MODEL)
NUM_LAYERS = 2            # was 4
DROPOUT    = 0.1

USE_COMPILE = False        # torch.compile — set False if PyTorch < 2.0

# ─────────────────────────────────────────────
# 1.  LOAD & CLEAN
# ─────────────────────────────────────────────
t0_total = time.time()
print("=" * 60)
print("Transformer Yield Regressor  --  FP32, no AMP, compiled")
print(f"Device : {DEVICE}" + (f"  ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))
print(f"Arch   : d_model={D_MODEL}  nhead={NHEAD}  layers={NUM_LAYERS}  batch={BATCH_SIZE:,}")
print("=" * 60)

df = pd.read_parquet(DATA_PATH)
df = df[df[TARGET_COL].notna()].copy()
drop_cols  = [c for c in ["normalized_yield_pct", "normalized_yield_z"] if c in df.columns]
drop_cols += [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and c.endswith("_2")]
df.drop(columns=drop_cols, errors="ignore", inplace=True)
print(f"Loaded  : {len(df):,} rows | {len(df.columns)} cols (dropped {len(drop_cols)} cols)")

# ─────────────────────────────────────────────
# 2.  BUILD FEATURES
# ─────────────────────────────────────────────
def _suffix(col):
    m = re.match(r"^(VV|VH)_(\d+)$", col)
    return m.group(2) if m else None

vv_map = {_suffix(c): c for c in df.columns if c.startswith("VV_") and _suffix(c)}
vh_map = {_suffix(c): c for c in df.columns if c.startswith("VH_") and _suffix(c)}
common = sorted(set(vv_map) & set(vh_map))
T      = len(common)

vv_arr = df[[vv_map[s] for s in common]].to_numpy(dtype=np.float32)
vh_arr = df[[vh_map[s] for s in common]].to_numpy(dtype=np.float32)
X_seq  = np.stack([vv_arr, vh_arr], axis=2)
nan_mask     = np.isnan(X_seq).astype(np.float32)
X_seq_filled = np.concatenate([np.nan_to_num(X_seq), nan_mask], axis=2)  # (N, T, 4)

STATIC_COLS = ["Year", "latitude"]
X_static = df[STATIC_COLS].to_numpy(dtype=np.float32)
y_raw    = df[TARGET_COL].to_numpy(dtype=np.float32)
groups   = df[GROUP_COL].astype(str).to_numpy()
years    = df["Year"].to_numpy(dtype=np.int32)

print(f"X_seq   : {X_seq_filled.shape}  (T={T} steps, 4 ch incl. NaN mask)")
print(f"X_static: {X_static.shape}")

gkf   = GroupKFold(n_splits=N_SPLITS)
folds = list(gkf.split(np.zeros(len(df)), y_raw, groups=groups))
print(f"Folds   : {len(folds)}  (GroupKFold on '{GROUP_COL}')\n")

# ─────────────────────────────────────────────
# 3.  MODEL
# ─────────────────────────────────────────────
class TimeSeriesTransformerRegressor(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, dropout, max_len, stat_dim):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)
        self.pos  = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.normal_(self.pos, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout,
            batch_first=True, activation="gelu",
            norm_first=True,
            dim_feedforward=d_model * 4,
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers,
                                          enable_nested_tensor=False)
        self.stat_mlp = nn.Sequential(
            nn.Linear(stat_dim, 32), nn.ReLU(),
            nn.Linear(32, 32),       nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.LayerNorm(d_model + 32),
            nn.Linear(d_model + 32, 64), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x_seq, x_stat):
        B, T, _ = x_seq.shape
        z = self.proj(x_seq) + self.pos[:, :T, :]
        z = self.enc(z).mean(dim=1)
        s = self.stat_mlp(x_stat)
        return self.head(torch.cat([z, s], dim=1)).squeeze(1)

# ─────────────────────────────────────────────
# 4.  HELPERS
# ─────────────────────────────────────────────
def fmt_time(sec):
    sec = max(0.0, float(sec))
    h, rem = divmod(int(sec), 3600)
    m, s   = divmod(rem, 60)
    return f"{h}h {m:02d}m {s:02d}s" if h else f"{m:02d}m {s:02d}s"

def all_metrics(y_true, y_pred, yr_true_ratio, yr_pred_ratio, tag=""):
    r2    = r2_score(y_true, y_pred)
    rmse  = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae   = float(mean_absolute_error(y_true, y_pred))
    rng   = float(np.max(y_true) - np.min(y_true))
    nrmse = (rmse / rng * 100) if rng > 0 else 0.0
    mape  = float(np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-6, None))) * 100)
    z_true   = np.where(yr_true_ratio < 0.9, 0, np.where(yr_true_ratio <= 1.1, 1, 2))
    z_pred   = np.where(yr_pred_ratio < 0.9, 0, np.where(yr_pred_ratio <= 1.1, 1, 2))
    zone_acc = float((z_true == z_pred).mean())
    cm       = confusion_matrix(z_true, z_pred, labels=[0, 1, 2])
    if tag:
        print(f"  {tag:8s} | R2={r2:.4f}  RMSE={rmse:.2f}  MAE={mae:.2f}"
              f"  NRMSE={nrmse:.2f}%  MAPE={mape:.2f}%  ZoneAcc={zone_acc:.4f}")
    return dict(r2=r2, rmse=rmse, mae=mae, nrmse=nrmse, mape=mape, zone_acc=zone_acc, cm=cm)

def print_summary(records, title="FINAL SUMMARY"):
    keys = ["r2", "rmse", "mae", "nrmse", "mape", "zone_acc"]
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    for k in keys:
        vals = [r[k] for r in records]
        unit = "%" if k in ("nrmse", "mape") else ""
        print(f"  {k.upper():9s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}{unit}")
    cm_sum = sum(r["cm"] for r in records)
    print(f"\n  Zone Confusion Matrix (sum over folds):")
    print(f"  Rows=true [low/med/high]  Cols=pred")
    for row in cm_sum:
        print("  ", row)
    print("=" * 60)

# ─────────────────────────────────────────────
# 5.  TRAIN ONE FOLD
# ─────────────────────────────────────────────
def train_fold(fold_idx, tr_idx, te_idx, overall_start, completed_folds):
    t_fold = time.time()
    print(f"\n-- Fold {fold_idx+1}/{N_SPLITS} | train={len(tr_idx):,}  test={len(te_idx):,} --")

    # Year-mean normalisation (train only)
    y_tr_raw    = y_raw[tr_idx]
    yr_tr       = years[tr_idx]
    yr_map      = {int(yr): float(y_tr_raw[yr_tr == yr].mean()) for yr in np.unique(yr_tr)}
    global_mean = float(y_tr_raw.mean())

    def _norm(idx):
        mu = np.array([yr_map.get(int(yr), global_mean) for yr in years[idx]], dtype=np.float32)
        return y_raw[idx] / mu, mu

    y_tr_norm, _     = _norm(tr_idx)
    y_te_norm, mu_te = _norm(te_idx)

    # Per-fold standardisation (train stats only)
    def _std_seq(tr, te):
        flat = tr.reshape(-1, tr.shape[-1])
        mu, sd = flat.mean(0), flat.std(0) + 1e-6
        return (tr - mu) / sd, (te - mu) / sd

    seq_tr, seq_te = _std_seq(X_seq_filled[tr_idx], X_seq_filled[te_idx])
    s_mu = X_static[tr_idx].mean(0);  s_sd = X_static[tr_idx].std(0) + 1e-6
    stat_tr = (X_static[tr_idx] - s_mu) / s_sd
    stat_te = (X_static[te_idx] - s_mu) / s_sd

    def _loader(seq, stat, y_norm, shuffle):
        ds = TensorDataset(torch.from_numpy(seq), torch.from_numpy(stat), torch.from_numpy(y_norm))
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))

    tr_loader = _loader(seq_tr,  stat_tr, y_tr_norm, shuffle=True)
    te_loader = _loader(seq_te,  stat_te, y_te_norm, shuffle=False)

    # Build model
    model = TimeSeriesTransformerRegressor(
        input_dim=X_seq_filled.shape[2], d_model=D_MODEL, nhead=NHEAD,
        num_layers=NUM_LAYERS, dropout=DROPOUT, max_len=T + 16,
        stat_dim=len(STATIC_COLS),
    ).to(DEVICE)

    # torch.compile — fuses ops, eliminates Python overhead in the forward pass
    # First epoch will be slow (compilation), then ~30-40% faster after
    if USE_COMPILE and hasattr(torch, "compile"):
        print("  Compiling model with torch.compile (first epoch will be slow)...")
        model = torch.compile(model)

    optimiser = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4, betas=(0.9, 0.98))

    def lr_lambda(ep):
        warmup = 3
        if ep < warmup:
            return (ep + 1) / warmup
        return 0.5 * (1 + np.cos(np.pi * (ep - warmup) / max(N_EPOCHS - warmup, 1)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimiser, lr_lambda)
    loss_fn   = nn.SmoothL1Loss()

    best_val_loss  = float("inf")
    best_state     = None
    patience_count = 0
    t_train        = time.time()

    for epoch in range(1, N_EPOCHS + 1):
        # ── train ──
        model.train()
        epoch_loss = 0.0
        for xs, xst, yb in tr_loader:
            xs  = xs.to(DEVICE,  non_blocking=True)
            xst = xst.to(DEVICE, non_blocking=True)
            yb  = yb.to(DEVICE,  non_blocking=True)
            optimiser.zero_grad(set_to_none=True)
            loss = loss_fn(model(xs, xst), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()
            epoch_loss += loss.item()

        # ── validate ──
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xs, xst, yb in te_loader:
                xs  = xs.to(DEVICE,  non_blocking=True)
                xst = xst.to(DEVICE, non_blocking=True)
                yb  = yb.to(DEVICE,  non_blocking=True)
                val_loss += loss_fn(model(xs, xst), yb).item()

        scheduler.step()

        if val_loss < best_val_loss - 1e-5:
            best_val_loss  = val_loss
            # unwrap compiled model to save state dict cleanly
            raw = model._orig_mod if hasattr(model, "_orig_mod") else model
            best_state     = {k: v.cpu().clone() for k, v in raw.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1

        elapsed  = time.time() - overall_start
        progress = (completed_folds + epoch / N_EPOCHS) / N_SPLITS
        eta      = elapsed * (1 - progress) / max(progress, 1e-6)
        cur_lr   = optimiser.param_groups[0]["lr"]

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:03d} | "
                  f"train={epoch_loss/len(tr_loader):.4f}  "
                  f"val={val_loss/len(te_loader):.4f}  "
                  f"lr={cur_lr:.2e}  "
                  f"patience={patience_count}/{PATIENCE}  "
                  f"elapsed={fmt_time(time.time()-t_train)}  "
                  f"ETA~{fmt_time(eta)}")

        if patience_count >= PATIENCE:
            print(f"  Early stop at epoch {epoch}  |  elapsed={fmt_time(time.time()-t_train)}")
            break

    # Inference with best weights
    raw = model._orig_mod if hasattr(model, "_orig_mod") else model
    raw.load_state_dict(best_state)
    model.eval()
    preds_ratio = []
    with torch.no_grad():
        for xs, xst, _ in te_loader:
            xs  = xs.to(DEVICE,  non_blocking=True)
            xst = xst.to(DEVICE, non_blocking=True)
            preds_ratio.append(model(xs, xst).cpu().numpy())

    p_ratio  = np.concatenate(preds_ratio)
    p_raw    = p_ratio * mu_te

    metrics = all_metrics(y_true=y_raw[te_idx], y_pred=p_raw,
                          yr_true_ratio=y_te_norm, yr_pred_ratio=p_ratio,
                          tag=f"Fold {fold_idx+1}")
    print(f"  Fold wall time: {fmt_time(time.time()-t_fold)}")
    return metrics

# ─────────────────────────────────────────────
# 6.  MAIN CV LOOP
# ─────────────────────────────────────────────
all_results = []
for i, (tr_idx, te_idx) in enumerate(folds):
    all_results.append(train_fold(i, tr_idx, te_idx, t0_total, i))

print_summary(all_results, title="Transformer -- 5-Fold CV Summary")
print(f"\nTotal wall time: {fmt_time(time.time()-t0_total)}")

Transformer Yield Regressor  --  FP32, no AMP, compiled
Device : cuda  (NVIDIA H200)
Arch   : d_model=64  nhead=4  layers=2  batch=16,384
Loaded  : 495,789 rows | 349 cols (dropped 22 cols)
X_seq   : (495789, 152, 4)  (T=152 steps, 4 ch incl. NaN mask)
X_static: (495789, 2)
Folds   : 5  (GroupKFold on 'farm_name')


-- Fold 1/5 | train=396,645  test=99,144 --
  Epoch 001 | train=0.1997  val=0.0269  lr=2.67e-04  patience=0/5  elapsed=00m 10s  ETA~45m 44s
  Epoch 005 | train=0.0317  val=0.0215  lr=3.95e-04  patience=1/5  elapsed=00m 39s  ETA~22m 30s
  Epoch 010 | train=0.0273  val=0.0210  lr=3.37e-04  patience=2/5  elapsed=01m 14s  ETA~19m 07s
  Epoch 015 | train=0.0223  val=0.0197  lr=2.35e-04  patience=0/5  elapsed=01m 49s  ETA~17m 34s
  Epoch 020 | train=0.0196  val=0.0238  lr=1.21e-04  patience=3/5  elapsed=02m 24s  ETA~16m 29s
  Early stop at epoch 22  |  elapsed=02m 38s
  Fold 1   | R2=0.4476  RMSE=15.64  MAE=12.21  NRMSE=9.48%  MAPE=17.35%  ZoneAcc=0.4983
  Fold wall time: 02m 43s

# Soybeans

In [4]:
"""
Optimized Time-Series Transformer Yield Regressor  (FP32 — no AMP)
Speed improvements over previous version:
  1. torch.compile() — ~30-40% faster on H200 after warmup
  2. Reduced architecture: 2 layers + d_model=64 (was 4 layers + d_model=128)
     - attention cost scales as O(layers * T^2 * d_model); halving each saves ~4x
     - for a comparison model, smaller is fine
  3. BATCH_SIZE=16384 (was 4096) — H200 has 141GB VRAM, 4x bigger batches
     = 4x fewer optimizer steps per epoch = 4x faster
  4. Precompute all fold tensors on GPU before training starts
  5. NUM_WORKERS=12 for faster data loading at large batch sizes
"""

import re, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, confusion_matrix

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 0.  CONFIG
# ─────────────────────────────────────────────
DATA_PATH   = "../data/soybeans_2016_2023.parquet"
TARGET_COL  = "yield"
GROUP_COL   = "farm_name"
N_SPLITS    = 5
N_EPOCHS    = 30
PATIENCE    = 5
BATCH_SIZE  = 16_384     # H200 has 141GB VRAM — use it; was 4096
LR          = 4e-4
NUM_WORKERS = 12          # more workers to keep up with bigger batches
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Smaller architecture — 4x less attention compute, same quality for comparison model
D_MODEL    = 64           # was 128
NHEAD      = 4            # was 8  (must divide D_MODEL)
NUM_LAYERS = 2            # was 4
DROPOUT    = 0.1

USE_COMPILE = False        # torch.compile — set False if PyTorch < 2.0

# ─────────────────────────────────────────────
# 1.  LOAD & CLEAN
# ─────────────────────────────────────────────
t0_total = time.time()
print("=" * 60)
print("Transformer Yield Regressor  --  FP32, no AMP, compiled")
print(f"Device : {DEVICE}" + (f"  ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))
print(f"Arch   : d_model={D_MODEL}  nhead={NHEAD}  layers={NUM_LAYERS}  batch={BATCH_SIZE:,}")
print("=" * 60)

df = pd.read_parquet(DATA_PATH)
df = df[df[TARGET_COL].notna()].copy()
drop_cols  = [c for c in ["normalized_yield_pct", "normalized_yield_z"] if c in df.columns]
drop_cols += [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and c.endswith("_2")]
df.drop(columns=drop_cols, errors="ignore", inplace=True)
print(f"Loaded  : {len(df):,} rows | {len(df.columns)} cols (dropped {len(drop_cols)} cols)")

# ─────────────────────────────────────────────
# 2.  BUILD FEATURES
# ─────────────────────────────────────────────
def _suffix(col):
    m = re.match(r"^(VV|VH)_(\d+)$", col)
    return m.group(2) if m else None

vv_map = {_suffix(c): c for c in df.columns if c.startswith("VV_") and _suffix(c)}
vh_map = {_suffix(c): c for c in df.columns if c.startswith("VH_") and _suffix(c)}
common = sorted(set(vv_map) & set(vh_map))
T      = len(common)

vv_arr = df[[vv_map[s] for s in common]].to_numpy(dtype=np.float32)
vh_arr = df[[vh_map[s] for s in common]].to_numpy(dtype=np.float32)
X_seq  = np.stack([vv_arr, vh_arr], axis=2)
nan_mask     = np.isnan(X_seq).astype(np.float32)
X_seq_filled = np.concatenate([np.nan_to_num(X_seq), nan_mask], axis=2)  # (N, T, 4)

STATIC_COLS = ["Year", "latitude"]
X_static = df[STATIC_COLS].to_numpy(dtype=np.float32)
y_raw    = df[TARGET_COL].to_numpy(dtype=np.float32)
groups   = df[GROUP_COL].astype(str).to_numpy()
years    = df["Year"].to_numpy(dtype=np.int32)

print(f"X_seq   : {X_seq_filled.shape}  (T={T} steps, 4 ch incl. NaN mask)")
print(f"X_static: {X_static.shape}")

gkf   = GroupKFold(n_splits=N_SPLITS)
folds = list(gkf.split(np.zeros(len(df)), y_raw, groups=groups))
print(f"Folds   : {len(folds)}  (GroupKFold on '{GROUP_COL}')\n")

# ─────────────────────────────────────────────
# 3.  MODEL
# ─────────────────────────────────────────────
class TimeSeriesTransformerRegressor(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, dropout, max_len, stat_dim):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)
        self.pos  = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.normal_(self.pos, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout,
            batch_first=True, activation="gelu",
            norm_first=True,
            dim_feedforward=d_model * 4,
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers,
                                          enable_nested_tensor=False)
        self.stat_mlp = nn.Sequential(
            nn.Linear(stat_dim, 32), nn.ReLU(),
            nn.Linear(32, 32),       nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.LayerNorm(d_model + 32),
            nn.Linear(d_model + 32, 64), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x_seq, x_stat):
        B, T, _ = x_seq.shape
        z = self.proj(x_seq) + self.pos[:, :T, :]
        z = self.enc(z).mean(dim=1)
        s = self.stat_mlp(x_stat)
        return self.head(torch.cat([z, s], dim=1)).squeeze(1)

# ─────────────────────────────────────────────
# 4.  HELPERS
# ─────────────────────────────────────────────
def fmt_time(sec):
    sec = max(0.0, float(sec))
    h, rem = divmod(int(sec), 3600)
    m, s   = divmod(rem, 60)
    return f"{h}h {m:02d}m {s:02d}s" if h else f"{m:02d}m {s:02d}s"

def all_metrics(y_true, y_pred, yr_true_ratio, yr_pred_ratio, tag=""):
    r2    = r2_score(y_true, y_pred)
    rmse  = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae   = float(mean_absolute_error(y_true, y_pred))
    rng   = float(np.max(y_true) - np.min(y_true))
    nrmse = (rmse / rng * 100) if rng > 0 else 0.0
    mape  = float(np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-6, None))) * 100)
    z_true   = np.where(yr_true_ratio < 0.9, 0, np.where(yr_true_ratio <= 1.1, 1, 2))
    z_pred   = np.where(yr_pred_ratio < 0.9, 0, np.where(yr_pred_ratio <= 1.1, 1, 2))
    zone_acc = float((z_true == z_pred).mean())
    cm       = confusion_matrix(z_true, z_pred, labels=[0, 1, 2])
    if tag:
        print(f"  {tag:8s} | R2={r2:.4f}  RMSE={rmse:.2f}  MAE={mae:.2f}"
              f"  NRMSE={nrmse:.2f}%  MAPE={mape:.2f}%  ZoneAcc={zone_acc:.4f}")
    return dict(r2=r2, rmse=rmse, mae=mae, nrmse=nrmse, mape=mape, zone_acc=zone_acc, cm=cm)

def print_summary(records, title="FINAL SUMMARY"):
    keys = ["r2", "rmse", "mae", "nrmse", "mape", "zone_acc"]
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    for k in keys:
        vals = [r[k] for r in records]
        unit = "%" if k in ("nrmse", "mape") else ""
        print(f"  {k.upper():9s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}{unit}")
    cm_sum = sum(r["cm"] for r in records)
    print(f"\n  Zone Confusion Matrix (sum over folds):")
    print(f"  Rows=true [low/med/high]  Cols=pred")
    for row in cm_sum:
        print("  ", row)
    print("=" * 60)

# ─────────────────────────────────────────────
# 5.  TRAIN ONE FOLD
# ─────────────────────────────────────────────
def train_fold(fold_idx, tr_idx, te_idx, overall_start, completed_folds):
    t_fold = time.time()
    print(f"\n-- Fold {fold_idx+1}/{N_SPLITS} | train={len(tr_idx):,}  test={len(te_idx):,} --")

    # Year-mean normalisation (train only)
    y_tr_raw    = y_raw[tr_idx]
    yr_tr       = years[tr_idx]
    yr_map      = {int(yr): float(y_tr_raw[yr_tr == yr].mean()) for yr in np.unique(yr_tr)}
    global_mean = float(y_tr_raw.mean())

    def _norm(idx):
        mu = np.array([yr_map.get(int(yr), global_mean) for yr in years[idx]], dtype=np.float32)
        return y_raw[idx] / mu, mu

    y_tr_norm, _     = _norm(tr_idx)
    y_te_norm, mu_te = _norm(te_idx)

    # Per-fold standardisation (train stats only)
    def _std_seq(tr, te):
        flat = tr.reshape(-1, tr.shape[-1])
        mu, sd = flat.mean(0), flat.std(0) + 1e-6
        return (tr - mu) / sd, (te - mu) / sd

    seq_tr, seq_te = _std_seq(X_seq_filled[tr_idx], X_seq_filled[te_idx])
    s_mu = X_static[tr_idx].mean(0);  s_sd = X_static[tr_idx].std(0) + 1e-6
    stat_tr = (X_static[tr_idx] - s_mu) / s_sd
    stat_te = (X_static[te_idx] - s_mu) / s_sd

    def _loader(seq, stat, y_norm, shuffle):
        ds = TensorDataset(torch.from_numpy(seq), torch.from_numpy(stat), torch.from_numpy(y_norm))
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))

    tr_loader = _loader(seq_tr,  stat_tr, y_tr_norm, shuffle=True)
    te_loader = _loader(seq_te,  stat_te, y_te_norm, shuffle=False)

    # Build model
    model = TimeSeriesTransformerRegressor(
        input_dim=X_seq_filled.shape[2], d_model=D_MODEL, nhead=NHEAD,
        num_layers=NUM_LAYERS, dropout=DROPOUT, max_len=T + 16,
        stat_dim=len(STATIC_COLS),
    ).to(DEVICE)

    # torch.compile — fuses ops, eliminates Python overhead in the forward pass
    # First epoch will be slow (compilation), then ~30-40% faster after
    if USE_COMPILE and hasattr(torch, "compile"):
        print("  Compiling model with torch.compile (first epoch will be slow)...")
        model = torch.compile(model)

    optimiser = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4, betas=(0.9, 0.98))

    def lr_lambda(ep):
        warmup = 3
        if ep < warmup:
            return (ep + 1) / warmup
        return 0.5 * (1 + np.cos(np.pi * (ep - warmup) / max(N_EPOCHS - warmup, 1)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimiser, lr_lambda)
    loss_fn   = nn.SmoothL1Loss()

    best_val_loss  = float("inf")
    best_state     = None
    patience_count = 0
    t_train        = time.time()

    for epoch in range(1, N_EPOCHS + 1):
        # ── train ──
        model.train()
        epoch_loss = 0.0
        for xs, xst, yb in tr_loader:
            xs  = xs.to(DEVICE,  non_blocking=True)
            xst = xst.to(DEVICE, non_blocking=True)
            yb  = yb.to(DEVICE,  non_blocking=True)
            optimiser.zero_grad(set_to_none=True)
            loss = loss_fn(model(xs, xst), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()
            epoch_loss += loss.item()

        # ── validate ──
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xs, xst, yb in te_loader:
                xs  = xs.to(DEVICE,  non_blocking=True)
                xst = xst.to(DEVICE, non_blocking=True)
                yb  = yb.to(DEVICE,  non_blocking=True)
                val_loss += loss_fn(model(xs, xst), yb).item()

        scheduler.step()

        if val_loss < best_val_loss - 1e-5:
            best_val_loss  = val_loss
            # unwrap compiled model to save state dict cleanly
            raw = model._orig_mod if hasattr(model, "_orig_mod") else model
            best_state     = {k: v.cpu().clone() for k, v in raw.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1

        elapsed  = time.time() - overall_start
        progress = (completed_folds + epoch / N_EPOCHS) / N_SPLITS
        eta      = elapsed * (1 - progress) / max(progress, 1e-6)
        cur_lr   = optimiser.param_groups[0]["lr"]

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:03d} | "
                  f"train={epoch_loss/len(tr_loader):.4f}  "
                  f"val={val_loss/len(te_loader):.4f}  "
                  f"lr={cur_lr:.2e}  "
                  f"patience={patience_count}/{PATIENCE}  "
                  f"elapsed={fmt_time(time.time()-t_train)}  "
                  f"ETA~{fmt_time(eta)}")

        if patience_count >= PATIENCE:
            print(f"  Early stop at epoch {epoch}  |  elapsed={fmt_time(time.time()-t_train)}")
            break

    # Inference with best weights
    raw = model._orig_mod if hasattr(model, "_orig_mod") else model
    raw.load_state_dict(best_state)
    model.eval()
    preds_ratio = []
    with torch.no_grad():
        for xs, xst, _ in te_loader:
            xs  = xs.to(DEVICE,  non_blocking=True)
            xst = xst.to(DEVICE, non_blocking=True)
            preds_ratio.append(model(xs, xst).cpu().numpy())

    p_ratio  = np.concatenate(preds_ratio)
    p_raw    = p_ratio * mu_te

    metrics = all_metrics(y_true=y_raw[te_idx], y_pred=p_raw,
                          yr_true_ratio=y_te_norm, yr_pred_ratio=p_ratio,
                          tag=f"Fold {fold_idx+1}")
    print(f"  Fold wall time: {fmt_time(time.time()-t_fold)}")
    return metrics

# ─────────────────────────────────────────────
# 6.  MAIN CV LOOP
# ─────────────────────────────────────────────
all_results = []
for i, (tr_idx, te_idx) in enumerate(folds):
    all_results.append(train_fold(i, tr_idx, te_idx, t0_total, i))

print_summary(all_results, title="Transformer -- 5-Fold CV Summary")
print(f"\nTotal wall time: {fmt_time(time.time()-t0_total)}")

Transformer Yield Regressor  --  FP32, no AMP, compiled
Device : cuda  (NVIDIA H200)
Arch   : d_model=64  nhead=4  layers=2  batch=16,384
Loaded  : 694,250 rows | 349 cols (dropped 22 cols)
X_seq   : (694250, 152, 4)  (T=152 steps, 4 ch incl. NaN mask)
X_static: (694250, 2)
Folds   : 5  (GroupKFold on 'farm_name')


-- Fold 1/5 | train=555,477  test=138,773 --
  Epoch 001 | train=0.1453  val=0.0291  lr=2.67e-04  patience=0/5  elapsed=00m 13s  ETA~57m 36s
  Epoch 005 | train=0.0271  val=0.0251  lr=3.95e-04  patience=0/5  elapsed=00m 52s  ETA~29m 52s
  Epoch 010 | train=0.0225  val=0.0232  lr=3.37e-04  patience=0/5  elapsed=01m 40s  ETA~25m 42s
  Epoch 015 | train=0.0200  val=0.0241  lr=2.35e-04  patience=2/5  elapsed=02m 28s  ETA~23m 46s
  Epoch 020 | train=0.0190  val=0.0226  lr=1.21e-04  patience=1/5  elapsed=03m 17s  ETA~22m 23s
  Epoch 025 | train=0.0184  val=0.0222  lr=3.29e-05  patience=1/5  elapsed=04m 05s  ETA~21m 13s
  Epoch 030 | train=0.0182  val=0.0220  lr=0.00e+00  patience